# Notebook 1 — Préparation des données (MIMIC-IV-ECG, PTB-XL, LSAS-ECG)

**Réglages Kaggle (panneau de droite) :** *Accelerator* **None**, *Internet* **On**,
*Add Input* : le dataset du code.

Lancez avec **Save Version → Save & Run All**. Le notebook s'arrête de lui-même avant la
limite des 12 h et **sauvegarde ce qui est fait**. S'il affiche **« À REPRENDRE »** :
1. *Edit* → *Add Input* → *Your Work* → ajoutez la sortie de **ce même notebook** (dernière version) ;
2. *Save Version → Save & Run All*. Ce qui est déjà fait est recopié, le reste est complété.

Répétez jusqu'à **« PRÉPARATION TERMINÉE »**, puis *Output → New Dataset* (`ecg-fusion-cache`).
À chaque reprise, ne gardez en Input que la sortie de la version la plus récente.

In [ ]:
import glob, os, re, shutil
def _version(init):
    m = re.search(r'__version__\s*=\s*"([^"]+)"', open(init).read())
    return m.group(1) if m else "0"
# plusieurs copies du code peuvent être présentes (dataset du code, sorties d'anciens notebooks) :
# on prend la plus récente
hits = sorted(glob.glob("/kaggle/input/**/ecgfusion/__init__.py", recursive=True), key=_version)
assert hits, "Code introuvable : ajoutez le dataset contenant ecg_fusion.zip via « Add Input »."
src = os.path.dirname(os.path.dirname(hits[-1]))
shutil.rmtree("/tmp/ecg_fusion", ignore_errors=True)
shutil.copytree(src, "/tmp/ecg_fusion")   # hors /kaggle/working : le code n'encombre pas la sortie
os.chdir("/tmp/ecg_fusion")
!pip install -q wfdb
print(f"Code version {_version(hits[-1])} prêt (source : {src})")

In [ ]:
import glob, json, shutil, time
T0 = time.time()
SESSION_HOURS = 11.0    # limite Kaggle = 12 h ; 1 h de marge pour sauvegarder la sortie
N_MIMIC = 100_000       # fixé lors de la 1re session (les reprises gardent le même échantillon)
WORKERS, THREADS = 4, 16
RAW, CACHE = "/tmp/raw", "/kaggle/working/cache"
os.makedirs(RAW, exist_ok=True); os.makedirs(CACHE, exist_ok=True)

def hours_left():
    return SESSION_HOURS - (time.time() - T0) / 3600

def previous(name):
    """Cache de ce nom dans les Inputs (sortie d'une version précédente)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/cache/{name}/info.json", recursive=True))
    return os.path.dirname(hits[0]) if hits else None

def info(d):
    return json.load(open(f"{d}/info.json"))

for n in ["ptbxl", "lsas", "mimic"]:
    p = previous(n)
    print(f"{n:6s} :", f"trouvé dans les Inputs ({info(p).get('n_done', info(p)['n'])}/{info(p)['n']})" if p else "à construire")

## 1. PTB-XL (validation externe principale)

In [ ]:
prev = previous("ptbxl")
if prev and info(prev).get("complete", True):
    shutil.copytree(prev, f"{CACHE}/ptbxl", dirs_exist_ok=True); print("PTB-XL repris de la session précédente")
else:
    !wget -q -c -O {RAW}/ptbxl.zip https://physionet.org/content/ptb-xl/get-zip/1.0.3/
    !unzip -q -o {RAW}/ptbxl.zip -d {RAW}/ptbxl && rm {RAW}/ptbxl.zip
    !python scripts/prepare_data.py --dataset ptbxl --root {RAW}/ptbxl --workers 4 --out {CACHE}/ptbxl
    !rm -rf {RAW}/ptbxl
print(f"temps restant : {hours_left():.1f} h")

## 2. LSAS-ECG (validation externe secondaire)

In [ ]:
prev = previous("lsas")
if prev and info(prev).get("complete", True):
    shutil.copytree(prev, f"{CACHE}/lsas", dirs_exist_ok=True); print("LSAS repris de la session précédente")
else:
    !wget -q -c -O {RAW}/lsas.zip https://physionet.org/content/ecg-arrhythmia/get-zip/1.0.0/
    !unzip -q -o {RAW}/lsas.zip -d {RAW}/lsas && rm {RAW}/lsas.zip
    !python scripts/prepare_data.py --dataset lsas --root {RAW}/lsas --workers 4 --out {CACHE}/lsas
    !rm -rf {RAW}/lsas
print(f"temps restant : {hours_left():.1f} h")

## 3. MIMIC-IV-ECG (lecture à distance, reprenable)
Signaux lus sur le miroir AWS de PhysioNet (rapide, sans compte), sinon sur physionet.org.
Le débit (ECG/s) et le temps restant estimé s'affichent toutes les 2 minutes.

In [ ]:
prev = previous("mimic")
if prev and not os.path.exists(f"{prev}/done.npy"):
    prev = None   # cache d'une ancienne version du code : on repart de zéro
if prev and info(prev)["complete"]:
    shutil.copytree(prev, f"{CACHE}/mimic", dirs_exist_ok=True); print("MIMIC déjà complet : repris")
else:
    budget = max(hours_left() - 0.3, 0.2)   # 0,3 h pour recopier/sauvegarder
    if prev:
        print(f"REPRISE MIMIC depuis {prev} ({info(prev)['n_done']}/{info(prev)['n']}), budget {budget:.1f} h")
        cmd = (f"python scripts/prepare_data.py --dataset mimic --resume_from {prev} --workers {WORKERS} "
               f"--threads {THREADS} --max_hours {budget:.4f} --out {CACHE}/mimic")
        !{cmd}
    else:
        !mkdir -p {RAW}/mimic
        !wget -q -c -P {RAW}/mimic https://physionet.org/files/mimic-iv-ecg/1.0/record_list.csv
        !wget -q -c -P {RAW}/mimic https://physionet.org/files/mimic-iv-ecg/1.0/machine_measurements.csv
        print(f"NOUVELLE préparation MIMIC ({N_MIMIC} ECG), budget {budget:.1f} h")
        cmd = (f"python scripts/prepare_data.py --dataset mimic --remote --root {RAW}/mimic --max_records {N_MIMIC} "
               f"--workers {WORKERS} --threads {THREADS} --max_hours {budget:.4f} --out {CACHE}/mimic")
        !{cmd}

## 4. Bilan

In [ ]:
all_done = True
for d in ["mimic", "ptbxl", "lsas"]:
    p = f"{CACHE}/{d}"
    if not os.path.exists(f"{p}/info.json"):
        print(f"{d:6s}  ABSENT"); all_done = False; continue
    i = info(p)
    complete = i.get("complete", True)
    all_done &= complete
    print(f"{d:6s}  {i.get('n_done', i['n'])}/{i['n']} traités, {i['n_valid']} valides  "
          f"{'complet' if complete else 'INCOMPLET'}")
    print("        prévalences :", {k: round(v, 3) for k, v in i["label_prevalence"].items()})
    if d == "mimic" and not complete:
        rate = i.get("last_session_rate_per_s", 0)
        left = i["n"] - i["n_done"]
        print(f"        débit {rate} ECG/s -> reste ~{left / max(rate, 1e-6) / 3600:.1f} h de calcul")
!du -sh {CACHE}/*
print("=" * 60)
print("PRÉPARATION TERMINÉE : créez le dataset depuis l'onglet Output." if all_done else
      "À REPRENDRE : ajoutez la sortie de cette version en Input puis relancez Save & Run All.")
print("=" * 60)

## 5. Audit des labels MIMIC
Phrases des comptes rendus qui déclenchent chaque label : copiez cette sortie pour vérification.

In [ ]:
if os.path.exists(f"{CACHE}/mimic/meta.csv"):
    !python scripts/audit_labels.py --cache {CACHE}/mimic --top 10